In [7]:
# ==========================================
# CELL 1: INFRASTRUCTURE REPAIR & DATA
# ==========================================
import os
import json

# 1. KILL THE BROKEN LIBRARIES
# We uninstall the pre-installed buggy versions
print("🧹 Cleaning up broken environment...")
!pip uninstall -y kaggle kagglesdk

# 2. INSTALL STABLE VERSIONS
# We force-install a specific version known to work
print("🛠️ Installing stable dependencies...")
!pip install -q "kaggle==1.5.16"
!pip install -q opendatasets gradio grad-cam

# 3. AUTHENTICATION (Manual Write)
# We create the credentials file manually to bypass environment variable bugs
print("🔑 Setting up credentials...")
creds = {"username": "suleimanraza", "key": "37f14f7b14865279286e19b7cff73a67"}

# Ensure .kaggle folder exists
!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)
!chmod 600 /root/.kaggle/kaggle.json

# 4. DOWNLOAD DATA
print("⬇️ Downloading Dataset...")
if not os.path.exists('dataset'):
    # We use the command line interface which is more robust than the python lib
    !kaggle datasets download -d sovitrath/diabetic-retinopathy-224x224-gaussian-filtered
    !unzip -q diabetic-retinopathy-224x224-gaussian-filtered.zip -d dataset/
    print("✅ Download Complete.")
else:
    print("✅ Dataset already exists.")

🧹 Cleaning up broken environment...
Found existing installation: kaggle 1.8.3
Uninstalling kaggle-1.8.3:
  Successfully uninstalled kaggle-1.8.3
Found existing installation: kagglesdk 0.1.14
Uninstalling kagglesdk-0.1.14:
  Successfully uninstalled kagglesdk-0.1.14
🛠️ Installing stable dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
🔑 Setting up credentials...
⬇️ Downloading Dataset...
100% 427M/427M [00:12<00:00, 42.1MB/s]
100% 427M/427M [00:12<00:00, 36.9MB/s]
✅ Download Complete.


In [8]:
# ==============================================================================
# PROJECT RETINA: FINAL INDEPENDENT PIPELINE
# Strategy: 3-Class Mapping | 512px Resolution | EfficientNet | Focal Loss
# ==============================================================================

import os
import copy
import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Imports that were installed in Cell 1
import gradio as gr
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# --- CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Computing on: {device}")
IMG_SIZE = 512
BATCH_SIZE = 12

# --- PHASE 1: DATA PREPARATION ---
print("\n--- PHASE 1: MAPPING TO 3 CLASSES @ 512px ---")
df = pd.read_csv('dataset/train.csv')
df['id_code'] = df['id_code'].apply(lambda x: x + ".png")

# Map 5 classes to 3 Clinical Categories
# 0 -> 0 (Healthy)
# 1,2 -> 1 (NPDR - Monitor)
# 3,4 -> 2 (Vision Threat - Refer)
label_map = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2}
df['diagnosis_3class'] = df['diagnosis'].map(label_map)

# Stratified Split
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['diagnosis_3class']
)

class RetinaDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.path_map = {}
        # Robust file finding
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith('.png'):
                    self.path_map[file] = os.path.join(root, file)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx, 0]
        label = self.dataframe.iloc[idx]['diagnosis_3class']
        img_path = self.path_map.get(img_name, os.path.join(self.root_dir, img_name))
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# Augmentation (Prevent Memorization)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset_root = 'dataset/gaussian_filtered_images/gaussian_filtered_images/'
train_dataset = RetinaDataset(train_df, root_dir=dataset_root, transform=train_transforms)
val_dataset = RetinaDataset(val_df, root_dir=dataset_root, transform=val_transforms)

# Weighted Sampler (To balance the batch)
class_counts = train_df['diagnosis_3class'].value_counts().sort_index().values
class_weights = 1. / class_counts
sample_weights = [class_weights[label] for label in train_df['diagnosis_3class']]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_df), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- PHASE 2: MODEL & FOCAL LOSS ---
print("\n--- PHASE 2: EFFICIENTNET & FOCAL LOSS ---")

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

# Load EfficientNet-B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
# Modify Head for 3 Classes
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 3)
model = model.to(device)

criterion = FocalLoss(gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# --- PHASE 3: TRAINING ---
print("\n--- PHASE 3: PROGRESSIVE TRAINING ---")

def train_progressive(model, loader, val_loader, criterion, optimizer, epochs=20):
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    # WARMUP (Backbone Frozen)
    print(">>> WARMUP (3 Epochs) <<<")
    for param in model.features.parameters():
        param.requires_grad = False

    for epoch in range(3):
        model.train()
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        print(f"  Warmup {epoch+1}: Loss {running_loss/len(loader.dataset):.4f}")

    # FINE TUNE (Unfreeze Top)
    print("\n>>> FINE TUNING (17 Epochs) <<<")
    for param in model.features[-3:].parameters():
        param.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=1e-5)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        # Validation
        model.eval()
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)

        val_acc = val_corrects.double() / len(val_loader.dataset)
        print(f"  Epoch {epoch+1}: Train Loss {running_loss/len(loader.dataset):.4f} | Val Acc {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
            print("    (Saved Best Model)")

    model.load_state_dict(best_wts)
    return model

# Run Training
model = train_progressive(model, train_loader, val_loader, criterion, optimizer, epochs=17)
torch.save(model.state_dict(), 'project_retina_final.pth')

# --- PHASE 4: AUDIT ---
print("\n--- PHASE 4: SAFETY AUDIT ---")
y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

class_names = ['Healthy', 'NPDR (Monitor)', 'Vision Threat (Refer)']
print(classification_report(y_true, y_pred, target_names=class_names))

# --- PHASE 5: APP ---
print("\n--- PHASE 5: LAUNCHING APP ---")
def predict_production(image):
    base_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.fromarray(image.astype('uint8'), 'RGB')

    # TTA (3 variations)
    variations = [img, img.transpose(Image.FLIP_LEFT_RIGHT), img.rotate(10)]
    batch = torch.stack([base_transform(v) for v in variations]).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(batch)
        probs = F.softmax(outputs, dim=1)
    avg_probs = torch.mean(probs, dim=0)

    p_healthy = float(avg_probs[0])
    p_npdr = float(avg_probs[1])
    p_vt = float(avg_probs[2])

    score_dict = {'Healthy': p_healthy, 'NPDR': p_npdr, 'Vision Threat': p_vt}

    # Safety Logic: P(VT) > 15% -> Refer
    if p_vt > 0.15:
        msg = f"⚠️ CRITICAL REFERRAL\nHigh Risk of Vision-Threatening DR ({p_vt:.1%})."
    elif p_healthy < 0.50:
        msg = f"⚠️ REFERRAL RECOMMENDED\nSigns of NPDR detected ({p_npdr:.1%})."
    else:
        msg = "✅ NEGATIVE\nNo significant signs of DR."

    # Grad-CAM
    input_tensor = base_transform(img).unsqueeze(0).to(device)
    target_layers = [model.features[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
    rgb_img = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    return score_dict, msg, visualization

interface = gr.Interface(
    fn=predict_production,
    inputs=gr.Image(label=f"Upload Fundus Photo ({IMG_SIZE}x{IMG_SIZE})"),
    outputs=[gr.Label(num_top_classes=3), gr.Textbox(label="Decision"), gr.Image(label="Heatmap")],
    title="Project Retina: Independent Edition",
    description="3-Class Clinical Model | EfficientNet | Focal Loss"
)
interface.launch(share=True, debug=True)

🚀 Computing on: cuda

--- PHASE 1: MAPPING TO 3 CLASSES @ 512px ---

--- PHASE 2: EFFICIENTNET & FOCAL LOSS ---
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 164MB/s]



--- PHASE 3: PROGRESSIVE TRAINING ---
>>> WARMUP (3 Epochs) <<<
  Warmup 1: Loss 0.3977
  Warmup 2: Loss 0.3103
  Warmup 3: Loss 0.2861

>>> FINE TUNING (17 Epochs) <<<
  Epoch 1: Train Loss 0.2460 | Val Acc 0.7558
    (Saved Best Model)
  Epoch 2: Train Loss 0.2249 | Val Acc 0.7940
    (Saved Best Model)
  Epoch 3: Train Loss 0.2008 | Val Acc 0.8063
    (Saved Best Model)
  Epoch 4: Train Loss 0.1798 | Val Acc 0.8226
    (Saved Best Model)
  Epoch 5: Train Loss 0.1799 | Val Acc 0.8254
    (Saved Best Model)
  Epoch 6: Train Loss 0.1674 | Val Acc 0.8363
    (Saved Best Model)
  Epoch 7: Train Loss 0.1583 | Val Acc 0.8281
  Epoch 8: Train Loss 0.1561 | Val Acc 0.8472
    (Saved Best Model)
  Epoch 9: Train Loss 0.1469 | Val Acc 0.8527
    (Saved Best Model)
  Epoch 10: Train Loss 0.1491 | Val Acc 0.8431
  Epoch 11: Train Loss 0.1560 | Val Acc 0.8377
  Epoch 12: Train Loss 0.1599 | Val Acc 0.8499
  Epoch 13: Train Loss 0.1413 | Val Acc 0.8540
    (Saved Best Model)
  Epoch 14: Train Los

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0a4f479804a0cc6b2c.gradio.live
